<a href="https://colab.research.google.com/github/Quantbrandon/CreditCart-SOM/blob/main/SOM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bibliotecas





In [ ]:
!pip -q install kagglehub

import kagglehub
import pandas as pd
import numpy as np

# Estandarización
from sklearn.preprocessing import StandardScaler

# Reducción de dimensionalidad
from sklearn.decomposition import PCA

# Gráficos interactivos
import plotly.graph_objects as go

# Parametros del SOM

In [ ]:
# Cantidad de filas de la malla
nrows = 5

# Cantidad de colunas de la malla
ncols = 5

# Cantidad de iteraciones
niters = 250

# Tasa de aprendizaje
learnig_rate = 0.5

# Radio de vecindad
rv = 2.0

# Preprocesado de datos

## Cargar datos

In [ ]:
# Direccion del data set
path = kagglehub.dataset_download(
    "arjunbhasin2013/ccdata"
)

# Cargar data set
df = pd.read_csv(
    f"{path}/CC GENERAL.csv"
)

# Eliminar ID
df = df.drop("CUST_ID", axis=1)

Using Colab cache for faster access to the 'ccdata' dataset.


## Eliminar NaN

In [ ]:
df = df.dropna()

## Estandarizacion

In [ ]:
x = StandardScaler().fit_transform(df)

# PCA

Para visualizar el proceso mediante el cual el SOM aprende y preserva la topología de los datos, se realizará una representación tridimensional del mapa. Para ello, se emplearán como ejes las tres primeras componentes principales obtenidas mediante un Análisis de Componentes Principales (PCA), permitiendo observar la evolución y organización de las neuronas en un espacio de menor dimensión.

In [ ]:
pca = PCA(n_components=3)

pca.fit(x)

plot_data = pca.transform(x)

print(
    "Varianza explicada por los primeros 3 componentes:",
    round(pca.explained_variance_ratio_.sum()*100, 2), "%"
)

Varianza explicada por los primeros 3 componentes: 56.52 %


# SOM

## Inicializacion del SOM

### Pesos iniciales

In [ ]:
# Cantidad de pesos aleatorios iniciales
n_pesos = x.shape[1]

# Generar de manera aletroa los pesos unason una distribucion uniforme ajusta a datos estandarizados
weights = np.random.uniform(
    low=x.min(axis=0),
    high=x.max(axis=0),
    size=(nrows, ncols, n_pesos)
)

## Funcion BMU

In [ ]:
def bmu(obs, weights):

    distances = np.linalg.norm(
        weights - obs,
        axis=2
    )

    return np.unravel_index(
        np.argmin(distances),
        distances.shape
    )

## Funcion de vecinidad

In [ ]:
def vecinidad(distance, rv):

    return np.exp(
        -(distance**2) /
        (2*rv**2)
    )

## Entrenamineto del SOM

In [ ]:

# Almacenar iteraciones
history = []


history = []

for iteration in range(niters):

    # Seleccionar observacion aleatoria
    sample = x[np.random.randint(len(x))]

    # Encontrar BMU
    bmu_row, bmu_col = bmu(
        sample,
        weights
    )

    # Actualizacion de neuronas

    for r in range(nrows):

        for c in range(ncols):

            # Distancia en la malla SOM

            grid_distance = np.sqrt(
                (r-bmu_row)**2 +
                (c-bmu_col)**2
            )

            # Influencia del BMU

            h = vecinidad(
                grid_distance,
                rv
            )

            # Actualizar pesos

            weights[r,c] += (
                learnig_rate *
                h *
                (
                    sample -
                    weights[r,c]
                )
            )

    # Guardar estado para animación posterior
    history.append({

        "weights":
            weights.copy(),

        "sample":
            sample.copy(),

        "bmu":
            (bmu_row,bmu_col)

    })


# Animacion

In [ ]:
# Almacenar los frames, cada frame es una iteracion
frames = []

for i, state in enumerate(history):

    W = state["weights"]         # pesos
    sample = state["sample"]     # observacion
    bmu_r, bmu_c = state["bmu"]  # coordenadas del BMU en la malla

    # Proyeccion PCA de los pesos
    W_pca = pca.transform(
        W.reshape(-1, n_pesos)
    ).reshape(nrows, ncols, 3)

    sample_pca = pca.transform(
        sample.reshape(1, -1)
    )[0]

    # Malla
    mesh_x, mesh_y, mesh_z = [], [], []

    # Conexiones por filas (líneas horizontales de la malla)
    for r in range(nrows):

        mesh_x.extend(W_pca[r, :, 0])
        mesh_y.extend(W_pca[r, :, 1])
        mesh_z.extend(W_pca[r, :, 2])

        # Separador para que Plotly no conecte filas entre sí
        mesh_x.append(None)
        mesh_y.append(None)
        mesh_z.append(None)

    # Conexiones por columnas (líneas verticales de la malla)
    for c in range(ncols):

        mesh_x.extend(W_pca[:, c, 0])
        mesh_y.extend(W_pca[:, c, 1])
        mesh_z.extend(W_pca[:, c, 2])

        # Separador entre columnas
        mesh_x.append(None)
        mesh_y.append(None)
        mesh_z.append(None)

    # Animacion
    frames.append(
        go.Frame(
            data=[

                # Datos originales
                go.Scatter3d(
                    x=plot_data[:, 0],
                    y=plot_data[:, 1],
                    z=plot_data[:, 2],
                    mode="markers",
                    marker=dict(size=2),
                    name="Clientes"
                ),

                # Malla del SOM proyectada
                go.Scatter3d(
                    x=mesh_x,
                    y=mesh_y,
                    z=mesh_z,
                    mode="lines+markers",
                    line=dict(width=4),
                    marker=dict(size=5),
                    name="Malla SOM"
                ),

                # Muestra actual usada en la iteración
                go.Scatter3d(
                    x=[sample_pca[0]],
                    y=[sample_pca[1]],
                    z=[sample_pca[2]],
                    mode="markers",
                    marker=dict(size=10, symbol="diamond"),
                    name="Dato actual"
                ),

                # BMU (Best Matching Unit)
                go.Scatter3d(
                    x=[W_pca[bmu_r, bmu_c, 0]],
                    y=[W_pca[bmu_r, bmu_c, 1]],
                    z=[W_pca[bmu_r, bmu_c, 2]],
                    mode="markers",
                    marker=dict(size=14, symbol="diamond"),
                    name="BMU"
                )
            ],
            name=str(i)  # identificador del frame
        )
    )


# Crear figura base con frames
fig = go.Figure(data=frames[0].data, frames=frames)


# Layout general del gráfico
fig.update_layout(
    title=f"SOM 3D {nrows}x{ncols}",
    width=1100,
    height=850,
    scene=dict(
        xaxis_title="PC1",
        yaxis_title="PC2",
        zaxis_title="PC3"
    ),

    # Botones de animación
    updatemenus=[dict(
        type="buttons",
        buttons=[
            dict(
                label="Play",
                method="animate",
                args=[None, {"frame": {"duration": 50, "redraw": True}}]
            ),
            dict(
                label="Pause",
                method="animate",
                args=[[None], {"mode": "immediate"}]
            )
        ]
    )]
)


# Slider de control de frames
steps = []

for k in range(len(frames)):
    steps.append(
        dict(
            method="animate",
            label=str(k),
            args=[
                [str(k)],
                {"mode": "immediate",
                 "frame": {"duration": 0, "redraw": True}}
            ]
        )
    )

fig.update_layout(
    sliders=[dict(
        currentvalue={"prefix": "Iteración: "},
        steps=steps
    )]
)



Buffered data was truncated after reaching the output size limit.

# Mostrar Animacion

In [ ]:

fig.show()

Buffered data was truncated after reaching the output size limit.